# Deep Learning-Based Phishing & Malicious Email Classifier

**Track 1 — Security Focus**

This Colab-ready notebook builds a PyTorch bidirectional LSTM classifier that accepts raw email text and predicts **Legitimate (0)** or **Phishing/Malicious (1)**. It generates a balanced synthetic dataset of 1,800 labeled emails, performs cleaning/tokenisation/padding, trains the model, and reports loss/accuracy curves, a confusion matrix, and F1-score.

**Runtime:** Google Colab → T4 GPU (GPU is optional for this small dataset).

In [ ]:
import re, random, json
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# Generate 1,800 synthetic labeled emails (900 per class).
phish_subjects=["Urgent account verification required","Security alert: unusual login","Your mailbox will be suspended","Payment failed - action required","Confirm your identity now","Important security notice","Password expiration warning","Unusual activity detected"]
phish_bodies=["We detected unusual activity on your account. Verify your identity immediately using the secure link below.","Your account will be locked within 24 hours unless you confirm your password and billing details.","A recent payment could not be processed. Sign in and update your card information to avoid suspension.","You have received a secure document. Open the attachment and enter your credentials to view it.","Our security team needs you to confirm your login information after a suspicious sign-in.","Failure to respond may result in permanent account suspension. Verify your details now."]
phish_calls=["verify now","confirm your account","update payment details","secure verification portal","click the link","validate your credentials"]
leg_subjects=["Weekly team update","Meeting agenda for Friday","Project status report","Invoice for approved purchase","Lunch and learn invitation","Quarterly planning notes","Your travel itinerary","Document review request","Welcome to the project","System maintenance notice"]
leg_bodies=["Hi team, attached are the notes from our meeting. Please review them before tomorrow afternoon.","The project milestone is on track. Let me know if you have questions about the next sprint.","Please find the approved invoice attached. Finance can process it through the normal procurement workflow.","We are sharing the agenda for Friday's meeting. No action is required before the discussion.","The scheduled maintenance window is this Saturday from 10 PM to midnight. Services may be briefly unavailable.","Thanks for your help on the project. Please send your feedback in the usual internal channel."]
leg_calls=["Thanks","Best regards","Please review","for your information","attached for reference","let me know if you need anything"]
rows=[]
for _ in range(900):
    s=random.choice(phish_subjects); b=random.choice(phish_bodies); c=random.choice(phish_calls)
    n=random.choice(["","\nDo not delay.","\nThis message requires immediate attention.","\nReference: ticket "+str(random.randint(10000,99999))])
    rows.append([s+" - "+str(random.randint(1,99)),"Hello,\n"+b+"\nPlease "+c+". "+n,1])
for _ in range(900):
    s=random.choice(leg_subjects); b=random.choice(leg_bodies); c=random.choice(leg_calls)
    n=random.choice(["","\nSent from the company mail system.","\nReference: project "+random.choice(["ORION","ALPHA","DELTA"]),"\nHave a good day."])
    rows.append([s+" - "+str(random.randint(1,99)),"Hello,\n"+b+"\n"+c+". "+n,0])
random.shuffle(rows)
df=pd.DataFrame(rows,columns=["subject","body","label"])
df["text"]=df.subject+" "+df.body
print(df.shape); print(df.label.value_counts())
df.head()

In [ ]:
# Text preprocessing: lowercase, normalise URLs/emails/numbers, remove punctuation, tokenise by whitespace.
def clean(t):
    t=t.lower(); t=re.sub(r'http\S+|www\.\S+',' url ',t); t=re.sub(r'\S+@\S+',' email ',t); t=re.sub(r'\d+',' num ',t); t=re.sub(r'[^a-z0-9_ ]',' ',t); return re.sub(r'\s+',' ',t).strip()
df["clean"]=df.text.map(clean)
train_df,test_df=train_test_split(df,test_size=.20,stratify=df.label,random_state=SEED)
train_df,val_df=train_test_split(train_df,test_size=.125,stratify=train_df.label,random_state=SEED)
print(len(train_df),len(val_df),len(test_df))

In [ ]:
# Build vocabulary from training data only and pad/truncate to 80 tokens.
counter=Counter(w for t in train_df.clean for w in t.split())
word2idx={"<PAD>":0,"<UNK>":1}
for w,_ in counter.most_common(5000): word2idx[w]=len(word2idx)
MAX_LEN=80
def encode(text):
    ids=[word2idx.get(w,1) for w in text.split()][:MAX_LEN]
    return ids+[0]*(MAX_LEN-len(ids))
def tensorize(frame):
    return torch.tensor([encode(x) for x in frame.clean],dtype=torch.long), torch.tensor(frame.label.values,dtype=torch.float32)
Xtr,ytr=tensorize(train_df); Xv,yv=tensorize(val_df); Xte,yte=tensorize(test_df)

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self,vocab,emb=64,hid=64):
        super().__init__()
        self.emb=nn.Embedding(vocab,emb,padding_idx=0)
        self.lstm=nn.LSTM(emb,hid,batch_first=True,bidirectional=True)
        self.drop=nn.Dropout(.30)
        self.fc=nn.Linear(hid*2,1)
    def forward(self,x):
        e=self.emb(x); o,_=self.lstm(e)
        mask=(x!=0).unsqueeze(-1); o=o*mask
        pooled=o.sum(1)/mask.sum(1).clamp(min=1)
        return self.fc(self.drop(pooled)).squeeze(1)
model=LSTMClassifier(len(word2idx)).to(device)
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
criterion=nn.BCEWithLogitsLoss()
print(model)

In [ ]:
train_loader=DataLoader(TensorDataset(Xtr,ytr),batch_size=32,shuffle=True)
val_loader=DataLoader(TensorDataset(Xv,yv),batch_size=64)

def evaluate(loader):
    model.eval(); ls=[]; probs=[]; ys=[]
    with torch.no_grad():
        for x,y in loader:
            x,y=x.to(device),y.to(device); z=model(x); ls.append(criterion(z,y).item()); probs.extend(torch.sigmoid(z).cpu().numpy()); ys.extend(y.cpu().numpy())
    pred=(np.array(probs)>=.5).astype(int)
    return np.mean(ls), accuracy_score(ys,pred), np.array(ys), pred

history={k:[] for k in ["loss","val_loss","accuracy","val_accuracy"]}
best=1e9; best_state=None; bad=0
for epoch in range(12):
    model.train(); ls=[]; probs=[]; ys=[]
    for x,y in train_loader:
        x,y=x.to(device),y.to(device); optimizer.zero_grad(); z=model(x); loss=criterion(z,y); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        ls.append(loss.item()); probs.extend(torch.sigmoid(z).detach().cpu().numpy()); ys.extend(y.cpu().numpy())
    tr_loss=np.mean(ls); tr_acc=accuracy_score(ys,(np.array(probs)>=.5).astype(int)); vl,va,_,_=evaluate(val_loader)
    history["loss"].append(tr_loss); history["accuracy"].append(tr_acc); history["val_loss"].append(vl); history["val_accuracy"].append(va)
    if vl<best: best=vl; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; bad=0
    else: bad+=1
    print(f"Epoch {epoch+1}: loss={tr_loss:.4f}, acc={tr_acc:.4f}, val_loss={vl:.4f}, val_acc={va:.4f}")
    if bad>=3: break
model.load_state_dict(best_state)

In [ ]:
# Evaluation metrics required by the assignment.
test_loss,test_acc,y_true,y_pred=evaluate(DataLoader(TensorDataset(Xte,yte),batch_size=64))
precision,recall,f1,_=precision_recall_fscore_support(y_true,y_pred,average="binary",zero_division=0)
cm=confusion_matrix(y_true,y_pred)
print({"Accuracy":test_acc,"Precision":precision,"Recall":recall,"F1":f1,"Test Loss":test_loss})
ConfusionMatrixDisplay(cm,display_labels=["Legitimate","Phishing"]).plot(); plt.title("Confusion Matrix"); plt.show()

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(history["loss"],label="Train"); ax[0].plot(history["val_loss"],label="Validation"); ax[0].set_title("Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(history["accuracy"],label="Train"); ax[1].plot(history["val_accuracy"],label="Validation"); ax[1].set_title("Accuracy"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.show()

In [ ]:
# Raw email inference function.
def predict_email(raw_email):
    model.eval(); x=torch.tensor([encode(clean(raw_email))],dtype=torch.long).to(device)
    with torch.no_grad(): p=torch.sigmoid(model(x)).item()
    return {"prediction":"PHISHING / MALICIOUS" if p>=.5 else "LEGITIMATE","phishing_probability":round(p,4)}

example="Urgent security alert: verify your account password immediately using the secure verification link."
print(predict_email(example))

## Hyperparameters
- Vocabulary: up to 5,000 training-set tokens; `<PAD>=0`, `<UNK>=1`
- Sequence length: 80 tokens
- Embedding: 64 dimensions
- BiLSTM hidden size: 64 per direction
- Dropout: 0.30
- Optimizer: Adam, learning rate 0.001
- Batch size: 32; maximum 12 epochs; early stopping patience 3
- Loss: Binary Cross-Entropy with Logits

## Deployment idea
The trained model can be wrapped in a Flask/FastAPI endpoint. A Gmail-like simulator posts `{subject, body}` as raw text; the service preprocesses the message, performs inference, and returns the predicted category and phishing probability.